In [4]:
import json
import pandas as pd
import ast

# -----------------------------
# Load & Decode JSONL
# -----------------------------

path = "master_resumes.jsonl"   # <-- change this to your file name

def safe_decode(s):
    """Decode JSON lines that may be double or triple encoded."""
    s = s.strip()

    # Strip outer quotes if present
    if (s.startswith('"') and s.endswith('"')) or (s.startswith("'") and s.endswith("'")):
        s = s[1:-1]

    # Fix doubled quotes
    # s = s.replace('""', '"')

    # Attempt decoding multiple times
    for _ in range(4):
        try:
            obj = json.loads(s)
        except:
            try:
                obj = ast.literal_eval(s)
            except:
                break

        # If we finally get a dict or list → done
        if isinstance(obj, (dict, list)):
            return obj

        # If still encoded as string → decode again
        if isinstance(obj, str):
            s = obj
            continue

        break

    return None


objs = []
with open(path, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = safe_decode(line)
        if isinstance(obj, dict):
            objs.append(obj)


# -----------------------------
# Flattening Helpers
# -----------------------------

def flatten_experience(exp):
    out = []

    if isinstance(exp, list):
        for e in exp:
            if not isinstance(e, dict):
                continue

            # ---- Responsibilities (list of strings) ----
            resp = e.get("responsibilities", [])
            if isinstance(resp, list):
                out.extend([str(x) for x in resp])

            # ---- Technical Environment ----
            tech_env = e.get("technical_environment", {})
            if isinstance(tech_env, dict):

                # technologies
                tech = tech_env.get("technologies", [])
                if isinstance(tech, list):
                    out.extend([str(x) for x in tech])

                # methodologies
                meth = tech_env.get("methodologies", [])
                if isinstance(meth, list):
                    out.extend([str(x) for x in meth])

                # tools
                tools = tech_env.get("tools", [])
                if isinstance(tools, list):
                    out.extend([str(x) for x in tools])

    return ", ".join(out)

def get_job_titles(exp):
    out = []

    if isinstance(exp, list):
        for e in exp:
            if not isinstance(e, dict):
                continue

            # ---- Title ----
            title = e.get("title")
            if isinstance(title, str):
                out.append(title)

    return ", ".join(out)


def flatten_education(edu):
    out = []
    if isinstance(edu, list):
        for e in edu:
            if isinstance(e, dict):
                sec = e.get("degree", {})
                if isinstance(sec, dict):
                    for v in sec.values():
                        if isinstance(v, (str, int, float)):
                            out.append(str(v))
    return ", ".join(out)


def flatten_projects(proj):
    out = []
    if isinstance(proj, list):
        for p in proj:
            if isinstance(p, dict):
                for k in ["description", "technologies", "impact", "role"]:
                    v = p.get(k, "")
                    if isinstance(v, str):
                        out.append(v)
    return ", ".join(out)


def flatten_skills(sk):
    out = []

    # Loop through top-level categories (but skip "languages")
    for category, items in sk.items():
        if category == "languages":
            continue

        # Add category name
        out.append(category)

        # Items inside category should be a list
        if isinstance(items, list):
            for it in items:
                if isinstance(it, dict):
                    name = it.get("name")
                    if isinstance(name, str):
                        out.append(name)

        # Some levels like project_management contain nested lists in dicts
        elif isinstance(items, dict):
            for subcat, subitems in items.items():
                out.append(subcat)
                if isinstance(subitems, list):
                    for it in subitems:
                        if isinstance(it, dict):
                            name = it.get("name")
                            if isinstance(name, str):
                                out.append(name)

    return ", ".join(out)


# -----------------------------
# Build Final Clean Dataset
# -----------------------------

rows = []
for o in objs:

    pi = o.get("personal_info", {})
    summary = ""
    if isinstance(pi.get("summary"), dict):
        summary = pi["summary"].get("content", "")
    else:
        summary = pi.get("summary", "")

    # summary = pi.get("summary", "")
    skills = flatten_skills(o.get("skills", {}))
    exp = flatten_experience(o.get("experience", []))
    edu = flatten_education(o.get("education", []))
    proj = flatten_projects(o.get("projects", []))

    job_title = get_job_titles(o.get("experience", []))

    # Master text (for ML training)
    resume_text = " ".join([skills, exp, edu, proj])

    rows.append({
        # "resume_summary": summary,
        "skills": skills,
        "experience": exp,
        "education": edu,
        "projects": proj,
        "combined_text": resume_text,
        "job_title": job_title,
    })

df = pd.DataFrame(rows)

display(df.head())
print(df.shape)

# -----------------------------
# Save CSV
# -----------------------------
df.to_csv("resumes_clean_labeled(2).csv", index=False)
print("Saved resumes_clean_labeled(2).csv")

,skills,experience,education,projects,combined_text,job_title
0,"technical, programming_languages, Python, C++,...","Python programming, Python, Tensorflow, Numpy,...","ME, Computer Engineering, Unknown, B.E, Comput...","Unknown, Unknown, Unknown","technical, programming_languages, Python, C++,...",Python Developer
1,"technical, project_management, Project Executi...","Reporting to the GM-Operations, Reviewing SOW,...","B.E, Electronics, Unknown",Led system configuration and application devel...,"technical, project_management, Project Executi...","Operations Manager, Project Lead, Project Engi..."
2,"technical, programming_languages, C, SQL, PL/S...",Working on the DevOps team in Parkar Consultin...,"B.E., Not Provided, Not Provided",AES is Advanced Encryption Standard which is u...,"technical, programming_languages, C, SQL, PL/S...",DevOps Engineer
3,"technical, project_management, Project Executi...","Reporting to the GM-Operations, Reviewing SOW,...","B.E, Electronics, Unknown",Led system configuration and application devel...,"technical, project_management, Project Executi...","Operations Manager, Project Lead, Project Engi..."
4,"technical, programming_languages, Python, fram...",Working as a developer in the field of compute...,"Bachelor of Engineering, Computer, , HSC, , , ...",,"technical, programming_languages, Python, fram...",Python Developer


(4817, 6)
Saved resumes_clean_labeled(2).csv


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4817 entries, 0 to 4816
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   skills         4817 non-null   object
 1   experience     4817 non-null   object
 2   education      4817 non-null   object
 3   projects       4817 non-null   object
 4   combined_text  4817 non-null   object
 5   job_title      4817 non-null   object
dtypes: object(6)
memory usage: 225.9+ KB


In [7]:
import pandas as pd
import re
from sklearn.preprocessing import MultiLabelBinarizer

# ===============================================================
# 1. Load dataset
# ===============================================================
df = pd.read_csv("resumes_clean_labeled(2).csv")   # change path if needed


# ===============================================================
# 2. Controlled vocabulary (MERGED: all dev roles → software engineer)
# ===============================================================
VOCAB = {
    "software engineer": [
        "developer", "software", "java", "python", "javascript",
        "full stack", "frontend", "backend"
    ],
    "mobile developer": ["android", "ios", "mobile"],
    "ui/ux designer": ["ui", "ux", "designer", "design"],
    "devops engineer": ["devops", "ci/cd", "automation", "infrastructure"],
    "cloud engineer": ["cloud", "aws", "azure", "gcp"],
    "sre": ["site reliability", "sre"],
    "data analyst": ["data analyst", "analytics"],
    "data engineer": ["data engineer", "etl", "pipeline"],
    "data scientist": ["data science", "scientist"],
    "machine learning engineer": ["machine learning", "ml engineer"],
    "project manager": ["project manager", "project lead", "project"],
    "operations manager": ["operations manager", "operations"],
    "product manager": ["product manager"],
    "qa engineer": ["qa", "quality assurance", "testing"],
    "security engineer": ["security", "cyber"],
    "support engineer": ["support"],
    "trainer": ["trainer", "instructor"],
    "advocate": ["advocate", "legal"]
}


# ===============================================================
# 3. Normalization function
# ===============================================================
def normalize_title(text):
    if pd.isna(text):
        return []
    text = text.lower()

    # split by comma or slash
    parts = [p.strip() for p in re.split(r"[,/]", text) if p.strip()]
    results = set()

    for part in parts:
        for clean_label, keywords in VOCAB.items():
            if any(k in part for k in keywords):
                results.add(clean_label)

    return sorted(results)


# ===============================================================
# 4. Apply normalization → create job_labels
# ===============================================================
df["job_labels"] = df["job_title"].apply(normalize_title)


# ===============================================================
# 5. Multi-hot encoding
# ===============================================================
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df["job_labels"])

# Add multi-hot columns to dataframe (optional)
for i, label in enumerate(mlb.classes_):
    df[f"label_{label}"] = Y[:, i]


# ===============================================================
# 6. Save cleaned dataset
# ===============================================================
df.to_csv("resumes_clean_multilabel(2).csv", index=False)

print("Done! Saved to resumes_clean_multilabel(2).csv")
print("Label space:", mlb.classes_)
print("Multi-hot matrix shape:", Y.shape)
print(df[["job_title", "job_labels"]].head())

Done! Saved to resumes_clean_multilabel(2).csv
Label space: ['advocate' 'cloud engineer' 'data engineer' 'data scientist'
 'devops engineer' 'machine learning engineer' 'mobile developer'
 'operations manager' 'project manager' 'qa engineer' 'security engineer'
 'software engineer' 'sre' 'trainer']
Multi-hot matrix shape: (4817, 14)
                                           job_title  \
0                                   Python Developer   
1  Operations Manager, Project Lead, Project Engi...   
2                                    DevOps Engineer   
3  Operations Manager, Project Lead, Project Engi...   
4                                   Python Developer   

                              job_labels  
0                    [software engineer]  
1  [operations manager, project manager]  
2                      [devops engineer]  
3  [operations manager, project manager]  
4                    [software engineer]  


In [8]:
df

,skills,experience,education,projects,combined_text,job_title,job_labels,label_advocate,label_cloud engineer,label_data engineer,...,label_devops engineer,label_machine learning engineer,label_mobile developer,label_operations manager,label_project manager,label_qa engineer,label_security engineer,label_software engineer,label_sre,label_trainer
0,"technical, programming_languages, Python, C++,...","Python programming, Python, Tensorflow, Numpy,...","ME, Computer Engineering, Unknown, B.E, Comput...","Unknown, Unknown, Unknown","technical, programming_languages, Python, C++,...",Python Developer,[software engineer],0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,"technical, project_management, Project Executi...","Reporting to the GM-Operations, Reviewing SOW,...","B.E, Electronics, Unknown",Led system configuration and application devel...,"technical, project_management, Project Executi...","Operations Manager, Project Lead, Project Engi...","[operations manager, project manager]",0,0,0,...,0,0,0,1,1,0,0,0,0,0
2,"technical, programming_languages, C, SQL, PL/S...",Working on the DevOps team in Parkar Consultin...,"B.E., Not Provided, Not Provided",AES is Advanced Encryption Standard which is u...,"technical, programming_languages, C, SQL, PL/S...",DevOps Engineer,[devops engineer],0,0,0,...,1,0,0,0,0,0,0,0,0,0
3,"technical, project_management, Project Executi...","Reporting to the GM-Operations, Reviewing SOW,...","B.E, Electronics, Unknown",Led system configuration and application devel...,"technical, project_management, Project Executi...","Operations Manager, Project Lead, Project Engi...","[operations manager, project manager]",0,0,0,...,0,0,0,1,1,0,0,0,0,0
4,"technical, programming_languages, Python, fram...",Working as a developer in the field of compute...,"Bachelor of Engineering, Computer, , HSC, , , ...",NaN,"technical, programming_languages, Python, fram...",Python Developer,[software engineer],0,0,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4812,"technical, programming_languages, JavaScript, ...",Integrated third-party services into existing ...,"BSc, Computer Science, Solutions, BSc, Compute...",Designed and implemented a comprehensive solut...,"technical, programming_languages, JavaScript, ...",Solutions Architect,[],0,0,0,...,0,0,0,0,0,0,0,0,0,0
4813,"technical, programming_languages, Java, JavaSc...",Collaborated with cross-functional teams to de...,"MSc, Computer Science, Solutions",Designed and implemented a comprehensive solut...,"technical, programming_languages, Java, JavaSc...","Solutions Architect, Solutions Architect, Solu...",[],0,0,0,...,0,0,0,0,0,0,0,0,0,0
4814,"technical, programming_languages, Go, C#, Pyth...",Ensured application responsiveness and seamles...,"BSc, Computer Science, Solutions, MSc, Compute...",Designed and implemented a comprehensive solut...,"technical, programming_languages, Go, C#, Pyth...","Solutions Architect, Solutions Architect, Solu...",[],0,0,0,...,0,0,0,0,0,0,0,0,0,0
4815,"technical, programming_languages, Go, C#, Pyth...","Implemented RESTful APIs and microservices., U...","MSc, Computer Science, Solutions, MSc, Compute...",Designed and implemented a comprehensive solut...,"technical, programming_languages, Go, C#, Pyth...",Solutions Architect,[],0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
label_counts = df["job_labels"].explode().value_counts()
print("\nOriginal Label Distribution:\n", label_counts)


Original Label Distribution:
 job_labels
software engineer            2131
devops engineer               336
security engineer             307
mobile developer              300
data scientist                109
cloud engineer                106
machine learning engineer     106
qa engineer                   101
sre                           100
data engineer                 100
operations manager             14
project manager                10
trainer                         6
advocate                        2
Name: count, dtype: int64
